# Modelo DeBERTa-v3-base — Fine-tuning para clasificación multi-etiqueta
**Requiere GPU.** Instalar dependencias: `pip install transformers sentencepiece torch`

**¿Por qué DeBERTa-v3 y no DistilBERT?**
- DeBERTa usa *disentangled attention* (separa contenido y posición) → mejor comprensión semántica
- DeBERTa-v3 fue pre-entrenado con ELECTRA (replaced token detection) → representaciones más ricas
- En benchmarks GLUE/SuperGLUE supera a BERT-large con menos parámetros
- Ganancia esperada sobre DistilBERT: **+0.02 a +0.04 Macro ROC-AUC**


In [ ]:
import warnings; warnings.filterwarnings('ignore')

import ast, re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cpu':
    print("ADVERTENCIA: sin GPU el entrenamiento será muy lento (~6-10h). Usa Google Colab o Kaggle.")


## 1. Datos

In [ ]:
train = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip',
    encoding='UTF-8', index_col=0
)
test = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip',
    encoding='UTF-8', index_col=0
)

# Texto: title repetido para dar peso + plot completo
train['text'] = (train['title'].fillna('') + '. ' +
                 train['title'].fillna('') + '. ' +
                 train['plot'].fillna(''))
test['text']  = (test['title'].fillna('') + '. ' +
                 test['title'].fillna('') + '. ' +
                 test['plot'].fillna(''))

train['genres'] = train['genres'].apply(ast.literal_eval)
mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(train['genres'])

X_train, X_valid, y_train, y_valid = train_test_split(
    train['text'], y, test_size=0.20, random_state=42
)
print(f"Train: {len(X_train)}  |  Valid: {len(X_valid)}")
print(f"Clases: {list(mlb.classes_)}")


## 2. Tokenización

In [ ]:
# Instalar: pip install transformers sentencepiece
MODEL_NAME = 'microsoft/deberta-v3-base'
# Alternativa más rápida: 'microsoft/deberta-v3-small' (~0.01 menos AUC, 3x más rápido)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, max_len=512):
    return tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors='pt'
    )

print("Tokenizando entrenamiento …")
train_enc = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=512)
valid_enc = tokenizer(X_valid.tolist(), truncation=True, padding=True, max_length=512)
test_enc  = tokenizer(test['text'].tolist(), truncation=True, padding=True, max_length=512)


## 3. Dataset PyTorch

In [ ]:
class MovieDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_ds = MovieDataset(train_enc, y_train)
valid_ds = MovieDataset(valid_enc, y_valid)
test_ds  = MovieDataset(test_enc)


## 4. Modelo con pérdida ponderada por clase

Para géneros raros (`News`, `Short`, `Western`) la distribución de positivos/negativos
es muy desbalanceada. `pos_weight = (N_neg / N_pos)` por clase le dice al modelo que
un error en un positivo es `w` veces más costoso que en un negativo, mejorando el
ROC-AUC en clases minoritarias.


In [ ]:
# Pesos positivos por clase (para BCEWithLogitsLoss)
class_counts = y.sum(axis=0)
pos_weight   = torch.tensor(
    (len(y) - class_counts) / np.maximum(class_counts, 1),
    dtype=torch.float
)
print("pos_weight por clase:")
for g, w in zip(mlb.classes_, pos_weight):
    print(f"  {g:15s}: {w:.1f}")


In [ ]:
from transformers import AutoConfig
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

def load_deberta_fixed(model_name, num_labels, problem_type):
    """
    Carga deberta-v3 corrigiendo el desajuste gamma/beta → weight/bias.

    El checkpoint de HuggingFace usa nomenclatura antigua de PyTorch para LayerNorm:
      gamma → weight   (escala)
      beta  → bias     (desplazamiento)
    La versión actual de transformers espera weight/bias, por lo que from_pretrained
    deja todos los LayerNorm aleatoriamente inicializados → AUC ~0.50.
    Este wrapper carga el .safetensors en crudo, renombra las claves y las inyecta.
    """
    # 1. Crear arquitectura vacía desde config
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type=problem_type
    )
    model = AutoModelForSequenceClassification.from_config(config)

    # 2. Descargar y cargar pesos crudos
    sf_path  = hf_hub_download(model_name, 'model.safetensors')
    raw_sd   = load_file(sf_path, device='cpu')

    # 3. Renombrar gamma → weight, beta → bias
    fixed_sd = {
        k.replace('.gamma', '.weight').replace('.beta', '.bias'): v
        for k, v in raw_sd.items()
    }

    # 4. Cargar pesos (strict=False: classifier + pooler son nuevos — esperado)
    result = model.load_state_dict(fixed_sd, strict=False)

    # 5. Verificar que no queden LayerNorm sin inicializar
    bad = [k for k in result.missing_keys if 'LayerNorm' in k]
    assert not bad, f"LayerNorm sigue sin cargar: {bad}"

    new_layers = [k for k in result.missing_keys if 'classifier' in k or 'pooler' in k]
    print(f"✓ Pesos pre-entrenados cargados correctamente.")
    print(f"  Capas nuevas (inicializadas para fine-tuning): {new_layers}")
    return model


model = load_deberta_fixed(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type='multi_label_classification'
)

class WeightedTrainer(Trainer):
    """Trainer con BCEWithLogitsLoss ponderado por frecuencia de clase."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = F.binary_cross_entropy_with_logits(
            outputs.logits,
            labels,
            pos_weight=pos_weight.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

## 5. Entrenamiento

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    probs = np.nan_to_num(probs, nan=0.5)
    scores = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) > 1:
            scores.append(roc_auc_score(labels[:, i], probs[:, i]))
    return {'macro_auc': float(np.mean(scores)) if scores else 0.0}

# ── Precisión numérica ─────────────────────────────────────────────────────────
# deberta-v3 tiene parámetros internos en fp16 → PyTorch lanza
# "Attempting to unscale FP16 gradients" cuando fp16=True está activo.
# Fix: bf16 si la GPU lo soporta (Ampere+: A100, RTX 3090/4090),
#      fp32 puro en caso contrario (T4, V100, GTX).
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False
print(f"Precisión: {'bf16 (rápido)' if USE_BF16 else 'fp32 (compatible con cualquier GPU)'}")

# Calcular warmup_steps explícitamente (warmup_ratio deprecado en transformers ≥5.2)
BATCH_SIZE   = 8
EPOCHS       = 5
STEPS_EPOCH  = -(-len(X_train) // BATCH_SIZE)   # ceil division
TOTAL_STEPS  = STEPS_EPOCH * EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * 0.10)
print(f"steps/epoch={STEPS_EPOCH}  total={TOTAL_STEPS}  warmup={WARMUP_STEPS}")

training_args = TrainingArguments(
    output_dir            = './results_deberta',
    eval_strategy         = 'epoch',
    save_strategy         = 'epoch',
    learning_rate         = 1e-5,
    per_device_train_batch_size = BATCH_SIZE,   # reducir a 4 si hay OOM
    per_device_eval_batch_size  = 16,
    num_train_epochs      = EPOCHS,
    weight_decay          = 0.01,
    warmup_steps          = WARMUP_STEPS,       # reemplaza warmup_ratio (deprecado)
    load_best_model_at_end= True,
    metric_for_best_model = 'macro_auc',
    greater_is_better     = True,
    fp16                  = False,              # SIEMPRE False con deberta-v3
    bf16                  = USE_BF16,           # True solo si GPU Ampere+
    max_grad_norm         = 1.0,
    logging_steps         = 100,
    save_total_limit      = 2,
    report_to             = 'none',
)

trainer = WeightedTrainer(
    model          = model,
    args           = training_args,
    train_dataset  = train_ds,
    eval_dataset   = valid_ds,
    compute_metrics= compute_metrics,
    callbacks      = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Evaluación

In [ ]:
metrics = trainer.evaluate()
print(f"Validation Macro ROC-AUC: {metrics['eval_macro_auc']:.4f}")
print(f"Baseline DistilBERT:       0.8996")
print(f"Ganancia:                  +{metrics['eval_macro_auc'] - 0.8996:.4f}")


## 7. Guardar predicciones de validación y test

In [ ]:
# Predicciones de validación (para optimizar el ensamble)
val_out   = trainer.predict(valid_ds)
val_probs = torch.sigmoid(torch.tensor(val_out.predictions)).numpy()
np.save('val_preds_deberta.npy', val_probs)

# Predicciones en test completo
test_out   = trainer.predict(test_ds)
test_probs = torch.sigmoid(torch.tensor(test_out.predictions)).numpy()

cols = ['p_Action','p_Adventure','p_Animation','p_Biography','p_Comedy',
        'p_Crime','p_Documentary','p_Drama','p_Family','p_Fantasy',
        'p_Film-Noir','p_History','p_Horror','p_Music','p_Musical',
        'p_Mystery','p_News','p_Romance','p_Sci-Fi','p_Short',
        'p_Sport','p_Thriller','p_War','p_Western']

sub = pd.DataFrame(test_probs, index=test.index, columns=cols)
sub.to_csv('pred_deberta.csv', index_label='ID')
print("Guardado: pred_deberta.csv")
sub.head()


## 8. Re-entrenamiento final en TODOS los datos de entrenamiento

Antes de la entrega final, re-entrenar con el dataset completo (sin split de validación)
extrae el máximo de información disponible.


In [ ]:
full_enc  = tokenizer(train['text'].tolist(), truncation=True, padding=True, max_length=512)
full_ds   = MovieDataset(full_enc, y)

# Cargar modelo con el mismo fix gamma/beta
model_full = load_deberta_fixed(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type='multi_label_classification'
)

STEPS_EPOCH_FULL  = -(-len(train) // BATCH_SIZE)
WARMUP_STEPS_FULL = int(STEPS_EPOCH_FULL * EPOCHS * 0.10)

args_full = TrainingArguments(
    output_dir                  = './results_deberta_full',
    learning_rate               = 1e-5,
    per_device_train_batch_size = BATCH_SIZE,
    num_train_epochs            = EPOCHS,
    weight_decay                = 0.01,
    warmup_steps                = WARMUP_STEPS_FULL,
    fp16                        = False,
    bf16                        = USE_BF16,
    max_grad_norm               = 1.0,
    logging_steps               = 200,
    save_strategy               = 'no',
    report_to                   = 'none',
)

trainer_full = WeightedTrainer(
    model         = model_full,
    args          = args_full,
    train_dataset = full_ds,
)
trainer_full.train()

# Predicción final
test_out_full   = trainer_full.predict(test_ds)
test_probs_full = torch.sigmoid(torch.tensor(test_out_full.predictions)).numpy()

sub_full = pd.DataFrame(test_probs_full, index=test.index, columns=cols)
sub_full.to_csv('pred_deberta_full.csv', index_label='ID')
print("Submission final: pred_deberta_full.csv")
sub_full.head()